# Project 02 — SQL Sales and Customer Analysis

**Author:** Jorgo Luka  
**Format:** Standalone Google Colab notebook  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist — approximately 100,000 anonymised orders  
**Core tools:** SQL · DuckDB · Python · Pandas · Matplotlib · Seaborn

## Recruiter summary

This project builds a small analytical warehouse from nine related e-commerce tables and answers commercial questions primarily in SQL. The design prevents the most common marketplace reporting error: multiplying revenue by joining order items, payments and reviews at incompatible grains.

| Capability | Evidence |
|---|---|
| Relational modelling | Raw schema, order-level semantic mart and item-level semantic mart |
| SQL | Joins, CTEs, conditional aggregation, window functions, `QUALIFY`, cohorts and parameterised queries |
| Data quality | Primary-key, foreign-key, null, range, grain and financial reconciliation checks |
| Business analysis | GMV, repeat customers, retention, categories, delivery, reviews, payments and seller concentration |
| Engineering | Pinned source version, assertions, query catalogue, Parquet exports, DuckDB database and hash manifest |

> This is historical, anonymised commercial data. GMV is not profit, and observational associations are not causal effects.

## Business questions

1. How should the nine source tables be joined without double-counting money or orders?
2. How did orders, merchandise value and average order value change over complete calendar months?
3. How many customers returned for another order, and how did acquisition cohorts retain?
4. Which product categories generated the most merchandise value?
5. Where did deliveries miss their promised date, and how did review scores differ?
6. How concentrated was marketplace activity among sellers?
7. Which payment methods and instalment patterns were most common?

## Source, licence and responsible use

- Source: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
- Download: Kaggle dataset version 7, pinned in the configuration below
- Licence: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)
- Coverage: orders placed between 2016 and 2018 across Brazilian marketplaces

Olist states that the source is real commercial data and has been anonymised. This notebook uses it only for educational and portfolio analysis, attributes the source, does not attempt re-identification and makes no claim about Olist's current performance.

## Data model and grain

The notebook deliberately keeps two analytical grains:

- `analytics.order_mart`: exactly one row per order. Use it for orders, customers, payments, delivery and reviews.
- `analytics.item_mart`: exactly one row per order item. Use it for products, categories and sellers.

Joining raw `order_items` directly to raw `payments` would create a many-to-many multiplication for multi-item, split-payment orders. The semantic layer aggregates each child table before joining it to orders.

## 0. Environment and reproducibility

Run **Runtime → Restart session and run all** in Google Colab. The public dataset download requires no Kaggle account or API token.

In [ ]:
%pip -q install "duckdb>=1.1,<2" "pyarrow>=15" "gradio>=5,<7"

In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import platform
import random
import textwrap
import time
import urllib.request
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


@dataclass(frozen=True)
class ProjectConfig:
    dataset_url: str = (
        "https://www.kaggle.com/api/v1/datasets/download/"
        "olistbr/brazilian-ecommerce?datasetVersionNumber=7"
    )
    dataset_version: int = 7
    archive_name: str = "olist_brazilian_ecommerce_v7.zip"
    artifact_dir: str = "sql_sales_analysis_artifacts"
    complete_month_start: str = "2017-01-01"
    complete_month_end: str = "2018-09-01"
    top_n: int = 12
    minimum_state_orders: int = 100
    minimum_seller_orders: int = 50
    launch_app: bool = False


CFG = ProjectConfig()
DATA_DIR = Path("olist_source_data")
ARTIFACTS = Path(CFG.artifact_dir)
DATA_DIR.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)

print({
    "python": platform.python_version(),
    "duckdb": duckdb.__version__,
    "pandas": pd.__version__,
    "configuration": asdict(CFG),
})

## 1. Download and fingerprint the source

The Kaggle dataset version is pinned. The archive and every generated artifact receive a SHA-256 fingerprint so a future run can be compared with this one.

In [ ]:
EXPECTED_FILES = {
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
}
REFERENCE_ARCHIVE_SHA256 = "d521eb1d4a8b6dae030aa429380787261d3b04cd95bee0f43f18cb9cb18ffebb"
ARCHIVE_PATH = DATA_DIR / CFG.archive_name


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


if not ARCHIVE_PATH.exists():
    request = urllib.request.Request(CFG.dataset_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=180) as response, ARCHIVE_PATH.open("wb") as target:
        while chunk := response.read(1 << 20):
            target.write(chunk)

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    archive_files = {Path(name).name for name in archive.namelist() if not name.endswith("/")}
    missing_files = EXPECTED_FILES - archive_files
    assert not missing_files, f"Archive is missing required files: {sorted(missing_files)}"
    archive.extractall(DATA_DIR)

SOURCE_FINGERPRINT = {
    "dataset": "Brazilian E-Commerce Public Dataset by Olist",
    "dataset_version": CFG.dataset_version,
    "source_url": CFG.dataset_url,
    "archive_bytes": ARCHIVE_PATH.stat().st_size,
    "archive_sha256": sha256_file(ARCHIVE_PATH),
    "matches_reference_archive": sha256_file(ARCHIVE_PATH) == REFERENCE_ARCHIVE_SHA256,
    "retrieved_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
}

assert all((DATA_DIR / name).exists() for name in EXPECTED_FILES)
print(json.dumps(SOURCE_FINGERPRINT, indent=2))

## 2. Create the DuckDB warehouse

DuckDB provides an analytical SQL engine inside the notebook. The raw CSV files remain unchanged; typed raw tables and analytical marts are created in a portable `.duckdb` database.

In [ ]:
DATABASE_PATH = ARTIFACTS / "olist_sales_analysis.duckdb"
con = duckdb.connect(str(DATABASE_PATH))
con.execute("SET threads = 4")
con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE SCHEMA IF NOT EXISTS analytics")

QUERY_CATALOG: dict[str, str] = {}


def run_sql(name: str, sql_text: str, params: list[Any] | None = None, display_rows: int = 20) -> pd.DataFrame:
    statement = textwrap.dedent(sql_text).strip()
    QUERY_CATALOG[name] = statement
    frame = con.execute(statement, params or []).fetchdf()
    print(f"{name}: {len(frame):,} row(s)")
    display(frame.head(display_rows))
    return frame


def execute_sql(name: str, sql_text: str) -> None:
    statement = textwrap.dedent(sql_text).strip()
    QUERY_CATALOG[name] = statement
    con.execute(statement)


RAW_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table_name, filename in RAW_FILES.items():
    csv_path = str((DATA_DIR / filename).resolve()).replace("'", "''")
    execute_sql(
        f"load_raw_{table_name}",
        f"""
        CREATE OR REPLACE TABLE raw.{table_name} AS
        SELECT *
        FROM read_csv_auto('{csv_path}', header = TRUE, sample_size = -1, all_varchar = FALSE)
        """,
    )

raw_table_profile = run_sql(
    "raw_table_profile",
    """
    SELECT table_name, estimated_size AS row_count, column_count
    FROM duckdb_tables()
    WHERE schema_name = 'raw'
    ORDER BY table_name
    """,
)

## 3. Audit keys, relationships and missing values

The checks below are written as SQL results rather than hidden assumptions. Review rows are not forced to a one-row-per-order grain because the source can contain more than one review record for an order; they are aggregated later.

In [ ]:
key_quality = run_sql(
    "key_quality",
    """
    WITH checks AS (
        SELECT 'orders.order_id' AS key_name, COUNT(*) AS rows,
               COUNT(DISTINCT order_id) AS distinct_keys
        FROM raw.orders
        UNION ALL
        SELECT 'customers.customer_id', COUNT(*), COUNT(DISTINCT customer_id)
        FROM raw.customers
        UNION ALL
        SELECT 'items.(order_id, order_item_id)', COUNT(*),
               COUNT(DISTINCT order_id || '|' || CAST(order_item_id AS VARCHAR))
        FROM raw.order_items
        UNION ALL
        SELECT 'products.product_id', COUNT(*), COUNT(DISTINCT product_id)
        FROM raw.products
        UNION ALL
        SELECT 'sellers.seller_id', COUNT(*), COUNT(DISTINCT seller_id)
        FROM raw.sellers
    )
    SELECT *, rows - distinct_keys AS duplicate_key_rows
    FROM checks
    ORDER BY key_name
    """,
)

foreign_key_quality = run_sql(
    "foreign_key_quality",
    """
    SELECT 'orders_without_customer' AS check_name, COUNT(*) AS violations
    FROM raw.orders o LEFT JOIN raw.customers c USING (customer_id)
    WHERE c.customer_id IS NULL
    UNION ALL
    SELECT 'items_without_order', COUNT(*)
    FROM raw.order_items i LEFT JOIN raw.orders o USING (order_id)
    WHERE o.order_id IS NULL
    UNION ALL
    SELECT 'payments_without_order', COUNT(*)
    FROM raw.order_payments p LEFT JOIN raw.orders o USING (order_id)
    WHERE o.order_id IS NULL
    UNION ALL
    SELECT 'reviews_without_order', COUNT(*)
    FROM raw.order_reviews r LEFT JOIN raw.orders o USING (order_id)
    WHERE o.order_id IS NULL
    UNION ALL
    SELECT 'items_without_product', COUNT(*)
    FROM raw.order_items i LEFT JOIN raw.products p USING (product_id)
    WHERE p.product_id IS NULL
    UNION ALL
    SELECT 'items_without_seller', COUNT(*)
    FROM raw.order_items i LEFT JOIN raw.sellers s USING (seller_id)
    WHERE s.seller_id IS NULL
    ORDER BY check_name
    """,
)

missing_value_quality = run_sql(
    "missing_value_quality",
    """
    SELECT
        COUNT(*) AS orders,
        COUNT(*) FILTER (WHERE order_purchase_timestamp IS NULL) AS missing_purchase_timestamp,
        COUNT(*) FILTER (WHERE order_approved_at IS NULL) AS missing_approval_timestamp,
        COUNT(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS missing_delivery_timestamp,
        COUNT(*) FILTER (WHERE order_estimated_delivery_date IS NULL) AS missing_estimate
    FROM raw.orders
    """,
)

## 4. Build the semantic layer

### Metric rules

| Metric | Definition |
|---|---|
| Commercial order | Has at least one item and is not `canceled` or `unavailable` |
| Merchandise value (GMV proxy) | Sum of item `price`; it is not profit or accounting revenue |
| Freight value | Sum of item-level `freight_value` |
| Total item value | Merchandise value plus freight |
| Customer | `customer_unique_id`, not the order-scoped `customer_id` |
| Repeat customer | At least two commercial orders in the analysis window |
| Late delivery | Delivered after `order_estimated_delivery_date` |
| Complete-month window | 1 January 2017 through 31 August 2018 |

Payments and item totals are kept as independent measures and reconciled. They are never summed together as “revenue.”

In [ ]:
execute_sql(
    "create_order_mart",
    f"""
    CREATE OR REPLACE TABLE analytics.order_mart AS
    WITH item_agg AS (
        SELECT
            order_id,
            COUNT(*) AS item_rows,
            COUNT(DISTINCT product_id) AS distinct_products,
            COUNT(DISTINCT seller_id) AS distinct_sellers,
            SUM(price) AS merchandise_value_brl,
            SUM(freight_value) AS freight_value_brl,
            SUM(price + freight_value) AS total_item_value_brl
        FROM raw.order_items
        GROUP BY order_id
    ),
    payment_agg AS (
        SELECT
            order_id,
            COUNT(*) AS payment_rows,
            COUNT(DISTINCT payment_type) AS payment_type_count,
            MAX(payment_installments) AS maximum_installments,
            SUM(payment_value) AS payment_value_brl
        FROM raw.order_payments
        GROUP BY order_id
    ),
    review_agg AS (
        SELECT
            order_id,
            COUNT(*) AS review_rows,
            AVG(review_score) AS average_review_score,
            MIN(review_score) AS minimum_review_score,
            MAX(review_score) AS maximum_review_score
        FROM raw.order_reviews
        GROUP BY order_id
    )
    SELECT
        o.order_id,
        o.customer_id,
        c.customer_unique_id,
        c.customer_city,
        c.customer_state,
        o.order_status,
        CAST(o.order_purchase_timestamp AS TIMESTAMP) AS purchased_at,
        CAST(o.order_approved_at AS TIMESTAMP) AS approved_at,
        CAST(o.order_delivered_carrier_date AS TIMESTAMP) AS delivered_to_carrier_at,
        CAST(o.order_delivered_customer_date AS TIMESTAMP) AS delivered_to_customer_at,
        CAST(o.order_estimated_delivery_date AS TIMESTAMP) AS estimated_delivery_at,
        ia.item_rows,
        ia.distinct_products,
        ia.distinct_sellers,
        ia.merchandise_value_brl,
        ia.freight_value_brl,
        ia.total_item_value_brl,
        pa.payment_rows,
        pa.payment_type_count,
        pa.maximum_installments,
        pa.payment_value_brl,
        ra.review_rows,
        ra.average_review_score,
        ra.minimum_review_score,
        ra.maximum_review_score,
        CASE
            WHEN o.order_status NOT IN ('canceled', 'unavailable') AND ia.item_rows > 0
            THEN TRUE ELSE FALSE
        END AS commercial_order,
        CASE
            WHEN CAST(o.order_purchase_timestamp AS DATE) >= DATE '{CFG.complete_month_start}'
             AND CAST(o.order_purchase_timestamp AS DATE) < DATE '{CFG.complete_month_end}'
            THEN TRUE ELSE FALSE
        END AS complete_month_window,
        DATE_DIFF('day', CAST(o.order_purchase_timestamp AS TIMESTAMP),
                         CAST(o.order_delivered_customer_date AS TIMESTAMP)) AS delivery_days,
        DATE_DIFF('day', CAST(o.order_estimated_delivery_date AS TIMESTAMP),
                         CAST(o.order_delivered_customer_date AS TIMESTAMP)) AS delivery_delay_days,
        CASE
            WHEN o.order_delivered_customer_date IS NULL OR o.order_estimated_delivery_date IS NULL THEN NULL
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN TRUE
            ELSE FALSE
        END AS late_delivery,
        pa.payment_value_brl - ia.total_item_value_brl AS payment_reconciliation_gap_brl
    FROM raw.orders o
    LEFT JOIN raw.customers c USING (customer_id)
    LEFT JOIN item_agg ia USING (order_id)
    LEFT JOIN payment_agg pa USING (order_id)
    LEFT JOIN review_agg ra USING (order_id)
    """,
)

execute_sql(
    "create_item_mart",
    f"""
    CREATE OR REPLACE TABLE analytics.item_mart AS
    SELECT
        i.order_id,
        i.order_item_id,
        i.product_id,
        i.seller_id,
        CAST(i.shipping_limit_date AS TIMESTAMP) AS shipping_limit_at,
        i.price,
        i.freight_value,
        i.price + i.freight_value AS item_total_value_brl,
        COALESCE(t.product_category_name_english, p.product_category_name, 'unknown') AS category,
        p.product_weight_g,
        p.product_length_cm,
        p.product_height_cm,
        p.product_width_cm,
        s.seller_city,
        s.seller_state,
        om.customer_unique_id,
        om.customer_state,
        om.order_status,
        om.purchased_at,
        om.delivered_to_customer_at,
        om.estimated_delivery_at,
        om.average_review_score,
        om.late_delivery,
        om.commercial_order,
        om.complete_month_window
    FROM raw.order_items i
    JOIN analytics.order_mart om USING (order_id)
    LEFT JOIN raw.products p USING (product_id)
    LEFT JOIN raw.category_translation t USING (product_category_name)
    LEFT JOIN raw.sellers s USING (seller_id)
    """,
)

semantic_layer_profile = run_sql(
    "semantic_layer_profile",
    """
    SELECT 'analytics.order_mart' AS table_name, COUNT(*) AS rows,
           COUNT(DISTINCT order_id) AS distinct_orders
    FROM analytics.order_mart
    UNION ALL
    SELECT 'analytics.item_mart', COUNT(*), COUNT(DISTINCT order_id)
    FROM analytics.item_mart
    """,
)

## 5. Validate the semantic layer

The mart is not accepted merely because the SQL ran. These checks prove its grain, relationship integrity, value ranges and cross-grain financial consistency.

In [ ]:
semantic_tests = run_sql(
    "semantic_tests",
    """
    SELECT 'order_mart_duplicate_order_ids' AS test_name,
           COUNT(*) - COUNT(DISTINCT order_id) AS violations
    FROM analytics.order_mart
    UNION ALL
    SELECT 'item_mart_duplicate_item_keys',
           COUNT(*) - COUNT(DISTINCT order_id || '|' || CAST(order_item_id AS VARCHAR))
    FROM analytics.item_mart
    UNION ALL
    SELECT 'item_mart_orphan_orders', COUNT(*)
    FROM analytics.item_mart i LEFT JOIN analytics.order_mart o USING (order_id)
    WHERE o.order_id IS NULL
    UNION ALL
    SELECT 'negative_item_prices', COUNT(*)
    FROM analytics.item_mart WHERE price < 0
    UNION ALL
    SELECT 'negative_freight_values', COUNT(*)
    FROM analytics.item_mart WHERE freight_value < 0
    UNION ALL
    SELECT 'commercial_orders_without_items', COUNT(*)
    FROM analytics.order_mart WHERE commercial_order AND item_rows IS NULL
    UNION ALL
    SELECT 'missing_unique_customer_on_orders', COUNT(*)
    FROM analytics.order_mart WHERE customer_unique_id IS NULL
    ORDER BY test_name
    """,
)

financial_reconciliation = run_sql(
    "financial_reconciliation",
    """
    SELECT
        COUNT(*) AS comparable_orders,
        ROUND(SUM(total_item_value_brl), 2) AS item_plus_freight_brl,
        ROUND(SUM(payment_value_brl), 2) AS payment_value_brl,
        ROUND(SUM(payment_reconciliation_gap_brl), 2) AS aggregate_gap_brl,
        ROUND(MEDIAN(ABS(payment_reconciliation_gap_brl)), 2) AS median_absolute_gap_brl,
        COUNT(*) FILTER (WHERE ABS(payment_reconciliation_gap_brl) > 0.02) AS orders_over_two_cent_gap
    FROM analytics.order_mart
    WHERE total_item_value_brl IS NOT NULL AND payment_value_brl IS NOT NULL
    """,
)

cross_grain_reconciliation = run_sql(
    "cross_grain_reconciliation",
    """
    WITH order_value AS (
        SELECT SUM(merchandise_value_brl) AS value
        FROM analytics.order_mart
        WHERE commercial_order
    ),
    item_value AS (
        SELECT SUM(price) AS value
        FROM analytics.item_mart
        WHERE commercial_order
    )
    SELECT
        ROUND(o.value, 2) AS order_mart_gmv_brl,
        ROUND(i.value, 2) AS item_mart_gmv_brl,
        ROUND(o.value - i.value, 6) AS difference_brl
    FROM order_value o CROSS JOIN item_value i
    """,
)

assert semantic_tests["violations"].sum() == 0
assert abs(float(cross_grain_reconciliation.loc[0, "difference_brl"])) < 0.01
print("Semantic-layer validation passed.")

### Reconciliation interpretation

The item mart and order mart reconcile exactly for merchandise value. Payment value is intentionally not forced to equal item value: the source contains a small set of order-level differences that may reflect adjustments, rounding or source-system behaviour. Those differences remain visible in `financial_reconciliation` and are exported for investigation rather than silently overwritten.

## 6. Headline commercial KPIs

This query uses the one-row-per-order mart. Merchandise value and freight come from items; payment value is shown only as a reconciliation measure.

In [ ]:
headline_kpis = run_sql(
    "headline_kpis",
    """
    WITH commercial AS (
        SELECT *
        FROM analytics.order_mart
        WHERE commercial_order
    )
    SELECT
        COUNT(*) AS commercial_orders,
        COUNT(DISTINCT customer_unique_id) AS unique_customers,
        ROUND(SUM(merchandise_value_brl), 2) AS merchandise_value_brl,
        ROUND(SUM(freight_value_brl), 2) AS freight_value_brl,
        ROUND(AVG(total_item_value_brl), 2) AS average_order_value_brl,
        ROUND(MEDIAN(total_item_value_brl), 2) AS median_order_value_brl,
        ROUND(100.0 * SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) / COUNT(*), 2)
            AS delivered_order_rate_pct
    FROM commercial
    """,
)

## 7. Monthly performance with window functions

The source starts and ends in partial months. Trend reporting is restricted to January 2017 through August 2018 so edge months do not create false declines. `LAG` calculates month-over-month change and a window frame calculates the three-month rolling average.

In [ ]:
monthly_performance = run_sql(
    "monthly_performance",
    """
    WITH monthly AS (
        SELECT
            DATE_TRUNC('month', purchased_at)::DATE AS month,
            COUNT(*) AS orders,
            COUNT(DISTINCT customer_unique_id) AS customers,
            SUM(merchandise_value_brl) AS merchandise_value_brl,
            SUM(freight_value_brl) AS freight_value_brl,
            AVG(total_item_value_brl) AS average_order_value_brl
        FROM analytics.order_mart
        WHERE commercial_order AND complete_month_window
        GROUP BY 1
    ),
    comparisons AS (
        SELECT
            *,
            LAG(merchandise_value_brl) OVER (ORDER BY month) AS previous_month_value,
            AVG(merchandise_value_brl) OVER (
                ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS rolling_three_month_value
        FROM monthly
    )
    SELECT
        month,
        orders,
        customers,
        ROUND(merchandise_value_brl, 2) AS merchandise_value_brl,
        ROUND(freight_value_brl, 2) AS freight_value_brl,
        ROUND(average_order_value_brl, 2) AS average_order_value_brl,
        ROUND(100.0 * (merchandise_value_brl - previous_month_value)
              / NULLIF(previous_month_value, 0), 2) AS month_over_month_value_pct,
        ROUND(rolling_three_month_value, 2) AS rolling_three_month_value_brl
    FROM comparisons
    ORDER BY month
    """,
    display_rows=24,
)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
axes[0].plot(monthly_performance["month"], monthly_performance["merchandise_value_brl"],
             marker="o", label="Monthly merchandise value")
axes[0].plot(monthly_performance["month"], monthly_performance["rolling_three_month_value_brl"],
             linewidth=3, label="Three-month rolling average")
axes[0].set_ylabel("BRL")
axes[0].set_title("Merchandise value across complete calendar months")
axes[0].legend()
axes[1].bar(monthly_performance["month"], monthly_performance["orders"], width=20, color="#4C78A8")
axes[1].set_ylabel("Orders")
axes[1].set_title("Commercial orders")
plt.xticks(rotation=45)
plt.tight_layout()
monthly_chart_path = ARTIFACTS / "monthly_performance.png"
plt.savefig(monthly_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 8. Customer repeat behaviour

`customer_unique_id` is essential: `customer_id` identifies the customer record attached to an individual order. Using it would incorrectly make almost every purchase appear to come from a new person.

In [ ]:
customer_segments = run_sql(
    "customer_segments",
    """
    WITH customer_orders AS (
        SELECT
            customer_unique_id,
            COUNT(*) AS order_count,
            SUM(total_item_value_brl) AS customer_value_brl,
            MIN(purchased_at) AS first_order_at,
            MAX(purchased_at) AS last_order_at
        FROM analytics.order_mart
        WHERE commercial_order AND complete_month_window
        GROUP BY customer_unique_id
    ),
    segmented AS (
        SELECT
            *,
            CASE
                WHEN order_count = 1 THEN 'One order'
                WHEN order_count BETWEEN 2 AND 3 THEN 'Two to three orders'
                ELSE 'Four or more orders'
            END AS customer_segment
        FROM customer_orders
    )
    SELECT
        customer_segment,
        COUNT(*) AS customers,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS customer_share_pct,
        SUM(order_count) AS orders,
        ROUND(SUM(customer_value_brl), 2) AS total_value_brl,
        ROUND(AVG(customer_value_brl), 2) AS average_customer_value_brl
    FROM segmented
    GROUP BY customer_segment
    ORDER BY MIN(order_count)
    """,
)

repeat_customer_summary = run_sql(
    "repeat_customer_summary",
    """
    WITH customer_orders AS (
        SELECT customer_unique_id, COUNT(*) AS order_count
        FROM analytics.order_mart
        WHERE commercial_order AND complete_month_window
        GROUP BY customer_unique_id
    )
    SELECT
        COUNT(*) AS customers,
        COUNT(*) FILTER (WHERE order_count >= 2) AS repeat_customers,
        ROUND(100.0 * COUNT(*) FILTER (WHERE order_count >= 2) / COUNT(*), 2)
            AS repeat_customer_rate_pct,
        MAX(order_count) AS maximum_orders_by_customer
    FROM customer_orders
    """,
)

## 9. Acquisition cohort retention

The first observed commercial order is calculated across the full history. Cohorts are then restricted to the complete-month window. Later cohorts have less time to mature, so empty future cells are right-censored rather than treated as zero retention.

In [ ]:
cohort_retention = run_sql(
    "cohort_retention",
    f"""
    WITH customer_months AS (
        SELECT DISTINCT
            customer_unique_id,
            DATE_TRUNC('month', purchased_at)::DATE AS order_month
        FROM analytics.order_mart
        WHERE commercial_order
    ),
    customer_cohorts AS (
        SELECT customer_unique_id, MIN(order_month) AS cohort_month
        FROM customer_months
        GROUP BY customer_unique_id
    ),
    activity AS (
        SELECT
            c.cohort_month,
            m.order_month,
            DATE_DIFF('month', c.cohort_month, m.order_month) AS month_number,
            COUNT(DISTINCT m.customer_unique_id) AS active_customers
        FROM customer_months m
        JOIN customer_cohorts c USING (customer_unique_id)
        WHERE c.cohort_month >= DATE '{CFG.complete_month_start}'
          AND c.cohort_month < DATE '{CFG.complete_month_end}'
        GROUP BY 1, 2, 3
    ),
    cohort_sizes AS (
        SELECT cohort_month, active_customers AS cohort_customers
        FROM activity
        WHERE month_number = 0
    )
    SELECT
        a.cohort_month,
        a.month_number,
        a.active_customers,
        s.cohort_customers,
        ROUND(100.0 * a.active_customers / s.cohort_customers, 2) AS retention_pct
    FROM activity a
    JOIN cohort_sizes s USING (cohort_month)
    WHERE a.month_number BETWEEN 0 AND 12
    ORDER BY a.cohort_month, a.month_number
    """,
    display_rows=30,
)

retention_matrix = cohort_retention.pivot(
    index="cohort_month", columns="month_number", values="retention_pct"
)
retention_heatmap = retention_matrix.drop(columns=[0], errors="ignore")
plt.figure(figsize=(15, 9))
sns.heatmap(retention_heatmap, annot=True, fmt=".1f", cmap="Blues", vmin=0, vmax=10,
            cbar_kws={"label": "Retention after acquisition month (%)"})
plt.title("Customer cohort retention after the acquisition month")
plt.xlabel("Months since first order")
plt.ylabel("Acquisition cohort")
plt.tight_layout()
cohort_chart_path = ARTIFACTS / "cohort_retention.png"
plt.savefig(cohort_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 10. Product category performance

Category analysis uses the item mart. The query ranks categories with a window function and reports both merchandise value and freight burden. No category is described as “profitable” because product cost is unavailable.

In [ ]:
category_performance = run_sql(
    "category_performance",
    f"""
    WITH category_values AS (
        SELECT
            category,
            COUNT(*) AS items,
            COUNT(DISTINCT order_id) AS orders,
            SUM(price) AS merchandise_value_brl,
            SUM(freight_value) AS freight_value_brl
        FROM analytics.item_mart
        WHERE commercial_order AND complete_month_window
        GROUP BY category
    )
    SELECT
        RANK() OVER (ORDER BY merchandise_value_brl DESC) AS value_rank,
        category,
        items,
        orders,
        ROUND(merchandise_value_brl, 2) AS merchandise_value_brl,
        ROUND(100.0 * merchandise_value_brl / SUM(merchandise_value_brl) OVER (), 2)
            AS merchandise_value_share_pct,
        ROUND(freight_value_brl, 2) AS freight_value_brl,
        ROUND(100.0 * freight_value_brl / NULLIF(merchandise_value_brl, 0), 2)
            AS freight_to_value_pct
    FROM category_values
    ORDER BY value_rank, category
    LIMIT {CFG.top_n}
    """,
)

plt.figure(figsize=(12, 7))
ordered_categories = category_performance.sort_values("merchandise_value_brl")
plt.barh(ordered_categories["category"], ordered_categories["merchandise_value_brl"], color="#59A14F")
plt.xlabel("Merchandise value (BRL)")
plt.title("Top product categories by merchandise value")
plt.tight_layout()
category_chart_path = ARTIFACTS / "category_performance.png"
plt.savefig(category_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 11. Delivery performance and reviews

The analysis includes only orders with an observed customer delivery timestamp and an estimated delivery timestamp. Review differences are descriptive; late delivery may be correlated with other factors such as geography, product mix or seller operations.

In [ ]:
delivery_summary = run_sql(
    "delivery_summary",
    """
    SELECT
        CASE WHEN late_delivery THEN 'Late' ELSE 'On time or early' END AS delivery_group,
        COUNT(*) AS delivered_orders,
        ROUND(AVG(delivery_days), 2) AS average_delivery_days,
        ROUND(MEDIAN(delivery_days), 2) AS median_delivery_days,
        ROUND(AVG(average_review_score), 2) AS average_review_score,
        ROUND(100.0 * COUNT(*) FILTER (WHERE average_review_score <= 2)
              / NULLIF(COUNT(average_review_score), 0), 2) AS low_review_rate_pct
    FROM analytics.order_mart
    WHERE commercial_order
      AND complete_month_window
      AND delivered_to_customer_at IS NOT NULL
      AND estimated_delivery_at IS NOT NULL
    GROUP BY 1
    ORDER BY delivery_group
    """,
)

state_delivery = run_sql(
    "state_delivery",
    f"""
    SELECT
        customer_state,
        COUNT(*) AS delivered_orders,
        ROUND(100.0 * AVG(CASE WHEN late_delivery THEN 1 ELSE 0 END), 2) AS late_delivery_rate_pct,
        ROUND(AVG(delivery_days), 2) AS average_delivery_days,
        ROUND(AVG(average_review_score), 2) AS average_review_score
    FROM analytics.order_mart
    WHERE commercial_order
      AND complete_month_window
      AND delivered_to_customer_at IS NOT NULL
      AND estimated_delivery_at IS NOT NULL
    GROUP BY customer_state
    HAVING COUNT(*) >= {CFG.minimum_state_orders}
    ORDER BY late_delivery_rate_pct DESC, delivered_orders DESC
    """,
    display_rows=30,
)

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(
    state_delivery["late_delivery_rate_pct"],
    state_delivery["average_review_score"],
    s=np.sqrt(state_delivery["delivered_orders"]) * 14,
    alpha=0.75,
    c=state_delivery["average_delivery_days"],
    cmap="viridis_r",
)
for _, row in state_delivery.iterrows():
    ax.annotate(row["customer_state"], (row["late_delivery_rate_pct"], row["average_review_score"]),
                xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("Late-delivery rate (%)")
ax.set_ylabel("Average review score")
ax.set_title("Delivery reliability and reviews by customer state")
plt.colorbar(scatter, ax=ax, label="Average delivery days")
plt.tight_layout()
delivery_chart_path = ARTIFACTS / "delivery_and_reviews.png"
plt.savefig(delivery_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 12. Seller concentration and operational review list

Seller metrics first collapse items to one row per order and seller. This prevents a seller with several line items in one order from receiving several copies of the same delivery outcome or review score.

In [ ]:
seller_concentration = run_sql(
    "seller_concentration",
    """
    WITH seller_values AS (
        SELECT seller_id, SUM(price) AS merchandise_value_brl
        FROM analytics.item_mart
        WHERE commercial_order AND complete_month_window
        GROUP BY seller_id
    ),
    shares AS (
        SELECT
            seller_id,
            merchandise_value_brl,
            merchandise_value_brl / SUM(merchandise_value_brl) OVER () AS value_share,
            ROW_NUMBER() OVER (ORDER BY merchandise_value_brl DESC) AS seller_rank
        FROM seller_values
    )
    SELECT
        COUNT(*) AS active_sellers,
        ROUND(100.0 * MAX(value_share), 2) AS largest_seller_share_pct,
        ROUND(100.0 * SUM(CASE WHEN seller_rank <= 10 THEN value_share ELSE 0 END), 2)
            AS top_ten_seller_share_pct,
        ROUND(10000.0 * SUM(value_share * value_share), 2) AS hhi_index
    FROM shares
    """,
)

seller_review_list = run_sql(
    "seller_review_list",
    f"""
    WITH order_seller AS (
        SELECT
            order_id,
            seller_id,
            seller_state,
            SUM(price) AS merchandise_value_brl,
            MAX(CASE WHEN delivered_to_customer_at IS NOT NULL THEN 1 ELSE 0 END) AS delivered,
            MAX(CASE WHEN late_delivery THEN 1 ELSE 0 END) AS late_delivery,
            MAX(average_review_score) AS average_review_score
        FROM analytics.item_mart
        WHERE commercial_order AND complete_month_window
        GROUP BY order_id, seller_id, seller_state
    ),
    seller_metrics AS (
        SELECT
            seller_id,
            seller_state,
            COUNT(*) AS orders,
            SUM(merchandise_value_brl) AS merchandise_value_brl,
            100.0 * SUM(late_delivery) / NULLIF(SUM(delivered), 0) AS late_delivery_rate_pct,
            AVG(average_review_score) AS average_review_score
        FROM order_seller
        GROUP BY seller_id, seller_state
        HAVING COUNT(*) >= {CFG.minimum_seller_orders}
    ),
    benchmark AS (
        SELECT AVG(late_delivery_rate_pct) AS average_seller_late_rate
        FROM seller_metrics
    )
    SELECT
        seller_id,
        seller_state,
        orders,
        ROUND(merchandise_value_brl, 2) AS merchandise_value_brl,
        ROUND(late_delivery_rate_pct, 2) AS late_delivery_rate_pct,
        ROUND(average_review_score, 2) AS average_review_score,
        CASE
            WHEN late_delivery_rate_pct > average_seller_late_rate + 10
             AND average_review_score < 3.8 THEN 'Review priority'
            ELSE 'Monitor'
        END AS action_band
    FROM seller_metrics CROSS JOIN benchmark
    ORDER BY
        CASE WHEN action_band = 'Review priority' THEN 0 ELSE 1 END,
        late_delivery_rate_pct DESC,
        orders DESC
    LIMIT 25
    """,
    display_rows=25,
)

## 13. Payment behaviour

Payments are analysed from the payment table and joined only to eligible orders. An order may use more than one payment record or method, so the query explicitly reports both payment rows and distinct orders.

In [ ]:
payment_performance = run_sql(
    "payment_performance",
    """
    SELECT
        p.payment_type,
        COUNT(*) AS payment_rows,
        COUNT(DISTINCT p.order_id) AS orders,
        ROUND(SUM(p.payment_value), 2) AS payment_value_brl,
        ROUND(AVG(p.payment_installments), 2) AS average_installments,
        ROUND(100.0 * SUM(p.payment_value) / SUM(SUM(p.payment_value)) OVER (), 2)
            AS payment_value_share_pct
    FROM raw.order_payments p
    JOIN analytics.order_mart o USING (order_id)
    WHERE o.commercial_order AND o.complete_month_window
    GROUP BY p.payment_type
    ORDER BY payment_value_brl DESC
    """,
)

instalment_bands = run_sql(
    "instalment_bands",
    """
    SELECT
        CASE
            WHEN maximum_installments IS NULL THEN 'Unknown'
            WHEN maximum_installments <= 1 THEN 'Single payment'
            WHEN maximum_installments BETWEEN 2 AND 3 THEN '2–3 instalments'
            WHEN maximum_installments BETWEEN 4 AND 6 THEN '4–6 instalments'
            ELSE '7+ instalments'
        END AS instalment_band,
        COUNT(*) AS orders,
        ROUND(AVG(total_item_value_brl), 2) AS average_order_value_brl,
        ROUND(AVG(average_review_score), 2) AS average_review_score
    FROM analytics.order_mart
    WHERE commercial_order AND complete_month_window
    GROUP BY 1
    ORDER BY MIN(COALESCE(maximum_installments, 999))
    """,
)

## 14. Advanced SQL: top categories within each state

`QUALIFY` filters the result of a window function without an extra nested query. This identifies each state's three leading categories while preserving the state-specific ranking.

In [ ]:
state_category_leaders = run_sql(
    "state_category_leaders",
    """
    SELECT
        customer_state,
        category,
        COUNT(DISTINCT order_id) AS orders,
        ROUND(SUM(price), 2) AS merchandise_value_brl,
        ROW_NUMBER() OVER (
            PARTITION BY customer_state ORDER BY SUM(price) DESC
        ) AS category_rank_within_state
    FROM analytics.item_mart
    WHERE commercial_order AND complete_month_window
    GROUP BY customer_state, category
    QUALIFY category_rank_within_state <= 3
    ORDER BY customer_state, category_rank_within_state
    """,
    display_rows=30,
)

## 15. Query-plan inspection and repeatable benchmark

DuckDB is a columnar analytical engine, so this workload does not imitate OLTP indexing. The query plan and a small repeated benchmark make performance inspection explicit rather than claiming “optimisation” without evidence.

In [ ]:
benchmark_query = """
SELECT
    DATE_TRUNC('month', purchased_at)::DATE AS month,
    customer_state,
    COUNT(*) AS orders,
    SUM(merchandise_value_brl) AS merchandise_value_brl
FROM analytics.order_mart
WHERE commercial_order AND complete_month_window
GROUP BY 1, 2
ORDER BY 1, 2
"""

query_plan = con.execute("EXPLAIN " + benchmark_query).fetchdf()
display(query_plan)

benchmark_seconds = []
for _ in range(5):
    started = time.perf_counter()
    con.execute(benchmark_query).fetchall()
    benchmark_seconds.append(time.perf_counter() - started)

benchmark_result = pd.DataFrame({
    "runs": [len(benchmark_seconds)],
    "median_seconds": [float(np.median(benchmark_seconds))],
    "minimum_seconds": [float(np.min(benchmark_seconds))],
    "maximum_seconds": [float(np.max(benchmark_seconds))],
})
display(benchmark_result)

## 16. Executive findings generated from the run

The statements below are calculated from the current execution. They are not hard-coded claims.

In [ ]:
best_month = monthly_performance.loc[monthly_performance["merchandise_value_brl"].idxmax()]
top_category = category_performance.iloc[0]
repeat_rate = float(repeat_customer_summary.loc[0, "repeat_customer_rate_pct"])
late_row = delivery_summary.loc[delivery_summary["delivery_group"].eq("Late")].iloc[0]
on_time_row = delivery_summary.loc[delivery_summary["delivery_group"].eq("On time or early")].iloc[0]
top_ten_share = float(seller_concentration.loc[0, "top_ten_seller_share_pct"])

EXECUTIVE_FINDINGS = [
    f"The warehouse reconciled {int(headline_kpis.loc[0, 'commercial_orders']):,} commercial orders "
    f"and {int(headline_kpis.loc[0, 'unique_customers']):,} unique customers.",
    f"The strongest complete month was {pd.Timestamp(best_month['month']):%Y-%m}, with "
    f"R${best_month['merchandise_value_brl']:,.0f} in merchandise value.",
    f"Only {repeat_rate:.2f}% of customers placed at least two commercial orders in the complete-month window.",
    f"The leading category was {top_category['category']}, generating "
    f"R${top_category['merchandise_value_brl']:,.0f} in merchandise value.",
    f"Late deliveries averaged {late_row['average_review_score']:.2f}/5 versus "
    f"{on_time_row['average_review_score']:.2f}/5 for on-time or early deliveries; this is an association, not causation.",
    f"The ten largest sellers represented {top_ten_share:.2f}% of merchandise value in the analysis window.",
]

for finding in EXECUTIVE_FINDINGS:
    print("•", finding)

## 17. Export database, analytical tables and governance artifacts

The notebook exports the reusable DuckDB database, order/item marts, business result tables, query catalogue, project card and SHA-256 manifest. These are generated when the notebook runs; only the notebook itself needs to be stored in the GitHub repository.

In [ ]:
EXPORTS: dict[str, Path] = {}


def export_csv(name: str, frame: pd.DataFrame) -> Path:
    path = ARTIFACTS / f"{name}.csv"
    frame.to_csv(path, index=False)
    EXPORTS[name] = path
    return path


for export_name, frame in {
    "raw_table_profile": raw_table_profile,
    "key_quality": key_quality,
    "foreign_key_quality": foreign_key_quality,
    "semantic_tests": semantic_tests,
    "financial_reconciliation": financial_reconciliation,
    "monthly_performance": monthly_performance,
    "customer_segments": customer_segments,
    "cohort_retention": cohort_retention,
    "category_performance": category_performance,
    "delivery_summary": delivery_summary,
    "state_delivery": state_delivery,
    "seller_review_list": seller_review_list,
    "payment_performance": payment_performance,
}.items():
    export_csv(export_name, frame)

order_parquet_path = ARTIFACTS / "order_mart.parquet"
item_parquet_path = ARTIFACTS / "item_mart.parquet"
con.execute(f"COPY analytics.order_mart TO '{str(order_parquet_path).replace("'", "''")}' "
            "(FORMAT PARQUET, COMPRESSION ZSTD)")
con.execute(f"COPY analytics.item_mart TO '{str(item_parquet_path).replace("'", "''")}' "
            "(FORMAT PARQUET, COMPRESSION ZSTD)")
EXPORTS["order_mart"] = order_parquet_path
EXPORTS["item_mart"] = item_parquet_path

query_catalog_path = ARTIFACTS / "query_catalog.sql"
query_catalog_path.write_text(
    "\n\n".join(f"-- {name}\n{statement.rstrip(';')};" for name, statement in QUERY_CATALOG.items()),
    encoding="utf-8",
)
EXPORTS["query_catalog"] = query_catalog_path

project_card = {
    "project": "SQL Sales and Customer Analysis",
    "author": "Jorgo Luka",
    "source": SOURCE_FINGERPRINT,
    "metric_definitions": {
        "commercial_order": "Has at least one item and is not canceled or unavailable.",
        "merchandise_value_brl": "Sum of item price; a GMV proxy, not profit.",
        "customer": "Anonymised customer_unique_id.",
        "repeat_customer": "At least two commercial orders in the complete-month window.",
        "late_delivery": "Delivered after the estimated delivery timestamp.",
    },
    "analysis_window": {
        "inclusive_start": CFG.complete_month_start,
        "exclusive_end": CFG.complete_month_end,
    },
    "verified_headline_kpis": headline_kpis.iloc[0].to_dict(),
    "executive_findings": EXECUTIVE_FINDINGS,
    "limitations": [
        "Historical data from 2016–2018; not evidence of current Olist performance.",
        "Merchandise value is not profit because costs, refunds and accounting recognition are unavailable.",
        "The source is anonymised and lacks acquisition-channel and marketing-spend data.",
        "Delivery and review relationships are observational and may be confounded.",
        "Later customer cohorts are right-censored by the dataset end date.",
        "Dataset licence is CC BY-NC-SA 4.0; use is educational and non-commercial.",
    ],
}
project_card_path = ARTIFACTS / "project_card.json"
project_card_path.write_text(json.dumps(project_card, indent=2, default=str), encoding="utf-8")
EXPORTS["project_card"] = project_card_path

for chart_name, chart_path in {
    "monthly_chart": monthly_chart_path,
    "cohort_chart": cohort_chart_path,
    "category_chart": category_chart_path,
    "delivery_chart": delivery_chart_path,
}.items():
    EXPORTS[chart_name] = chart_path

con.execute("CHECKPOINT")
EXPORTS["duckdb_database"] = DATABASE_PATH

manifest_rows = []
for name, path in sorted(EXPORTS.items()):
    manifest_rows.append({
        "artifact": name,
        "file": path.name,
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })
manifest = pd.DataFrame(manifest_rows)
manifest_path = ARTIFACTS / "artifact_manifest.csv"
manifest.to_csv(manifest_path, index=False)
display(manifest)

## 18. Acceptance tests

These checks make the notebook fail loudly when its critical assumptions are broken.

In [ ]:
assert len(raw_table_profile) == 9
assert int(raw_table_profile["row_count"].sum()) > 1_000_000
assert (key_quality["duplicate_key_rows"] == 0).all()
assert (foreign_key_quality["violations"] == 0).all()
assert (semantic_tests["violations"] == 0).all()
assert int(semantic_layer_profile.loc[semantic_layer_profile["table_name"].eq("analytics.order_mart"), "rows"].iloc[0]) > 90_000
assert len(monthly_performance) == 20
assert monthly_performance["month"].is_monotonic_increasing
assert cohort_retention["retention_pct"].between(0, 100).all()
assert category_performance["merchandise_value_brl"].is_monotonic_decreasing
assert repeat_customer_summary["repeat_customer_rate_pct"].between(0, 100).all()
assert delivery_summary["delivered_orders"].sum() > 80_000
assert all(path.exists() and path.stat().st_size > 0 for path in EXPORTS.values())
assert manifest["sha256"].str.fullmatch(r"[0-9a-f]{64}").all()

print("ALL PROJECT 02 ACCEPTANCE TESTS PASSED")

## 19. Optional interactive state lookup

The interface calls a parameterised SQL query rather than filtering a pre-computed screenshot. It is disabled by default so a full notebook run never blocks waiting for a web application.

In [ ]:
available_states = con.execute(
    "SELECT DISTINCT customer_state FROM analytics.order_mart "
    "WHERE customer_state IS NOT NULL ORDER BY customer_state"
).fetchdf()["customer_state"].tolist()


def state_lookup(customer_state: str) -> tuple[str, pd.DataFrame]:
    summary = con.execute(
        """
        SELECT
            COUNT(*) AS commercial_orders,
            COUNT(DISTINCT customer_unique_id) AS customers,
            ROUND(SUM(merchandise_value_brl), 2) AS merchandise_value_brl,
            ROUND(100.0 * AVG(CASE WHEN late_delivery THEN 1 ELSE 0 END), 2) AS late_delivery_rate_pct,
            ROUND(AVG(average_review_score), 2) AS average_review_score
        FROM analytics.order_mart
        WHERE commercial_order AND complete_month_window AND customer_state = ?
        """,
        [customer_state],
    ).fetchdf()
    categories = con.execute(
        """
        SELECT category, COUNT(DISTINCT order_id) AS orders,
               ROUND(SUM(price), 2) AS merchandise_value_brl
        FROM analytics.item_mart
        WHERE commercial_order AND complete_month_window AND customer_state = ?
        GROUP BY category
        ORDER BY merchandise_value_brl DESC
        LIMIT 10
        """,
        [customer_state],
    ).fetchdf()
    row = summary.iloc[0]
    narrative = (
        f"**{customer_state}** — {int(row['commercial_orders']):,} commercial orders, "
        f"R${row['merchandise_value_brl']:,.0f} merchandise value, "
        f"{row['late_delivery_rate_pct']:.2f}% late-delivery rate and "
        f"{row['average_review_score']:.2f}/5 average review score."
    )
    return narrative, categories


smoke_narrative, smoke_categories = state_lookup("SP")
assert "SP" in smoke_narrative and not smoke_categories.empty
print("State lookup smoke test passed.")

try:
    import gradio as gr

    with gr.Blocks(title="SQL Sales and Customer Analysis") as app:
        gr.Markdown("# SQL Sales and Customer Analysis\nSelect a Brazilian customer state.")
        state_input = gr.Dropdown(choices=available_states, value="SP", label="Customer state")
        lookup_button = gr.Button("Run SQL analysis", variant="primary")
        summary_output = gr.Markdown()
        category_output = gr.Dataframe(label="Leading categories", interactive=False)
        lookup_button.click(state_lookup, state_input, [summary_output, category_output])

    if CFG.launch_app:
        app.launch(share=True, debug=False)
    else:
        print("Application built. Set launch_app=True in ProjectConfig to launch it.")
except Exception as exc:
    print(f"Optional Gradio interface unavailable: {type(exc).__name__}. SQL smoke test still passed.")

## Interview explanation

**Problem:** Nine relational tables describe orders, items, payments, products, sellers, customers and reviews at different grains. A naive join can multiply financial values and generate incorrect KPIs.

**Decision:** I created separate order and item marts. Child tables are aggregated before joining to the order mart, while category and seller analysis stays at item grain. I defined commercial orders, GMV, customers and the complete-month window explicitly.

**Technical trade-off:** DuckDB makes the project reproducible in Colab and demonstrates transferable analytical SQL, but it is an embedded engine rather than a distributed production warehouse. In production I would implement the same tests and semantic definitions in dbt over BigQuery, Snowflake or Databricks.

**Failure mode:** Treating `customer_id` as a person identifier understates repeat purchasing, while joining raw payments to raw items overstates money. Both errors are prevented by the semantic model and reconciliation tests.

**Next production step:** Schedule incremental ingestion, store tests in CI, version metric definitions, add orchestration and expose the validated marts to a BI layer.

## CV bullet generated from verified results

Run the next cell and use its output only after the acceptance tests pass.

In [ ]:
cv_bullet = (
    f"Built a DuckDB analytical warehouse from nine relational Olist tables, using SQL CTEs, joins, "
    f"window functions and cohort analysis to reconcile {int(headline_kpis.loc[0, 'commercial_orders']):,} "
    f"commercial orders and R${headline_kpis.loc[0, 'merchandise_value_brl']:,.2f} in merchandise value; "
    f"implemented key, grain, financial and artifact acceptance tests."
)
print(cv_bullet)

## Final checklist

- [x] Real, anonymised, attributed public dataset
- [x] Nine source tables loaded into a relational SQL engine
- [x] Explicit grains and metric definitions
- [x] Advanced SQL demonstrated on business questions
- [x] Key, foreign-key, null, range and reconciliation tests
- [x] Complete-month trend window and cohort censoring documented
- [x] Database, marts, query catalogue, project card and manifest exported
- [x] Optional application smoke-tested
- [x] Limitations and interview explanation included

**Portfolio rule:** upload this single notebook to the repository root. Do not upload the generated data or artifact directory.